# DINOv2 с регистрами (B/L) для двух верификаторов: городища (виньетки SpOr) и архитектура Or (кроп-верификатор) — bot-5
**Только честный замер (OOF по семьям), не сдача.** Запуск — bot 9. GPU T4 ×1, Internet ON.
Inputs: репо `expds-koz3-bot5` (ветка `bot5-dinov2reg`), `train-expds-3`. Для части B нужны ещё стек до порогов и эталон (пути — STACK/GT).
Веса (HF, apache-2.0, ревизии пиннуты 29.10.2026): `facebook/dinov2-with-registers-base@a1d738ccfa7ae170945f210395d99dde8adb1805`,
`facebook/dinov2-with-registers-large@e4c89a4e05589de9b3e188688a303d0f3c04d0f3`.

In [ ]:
REPO='/kaggle/input/expds-koz3-bot5'   # поправить на фактический путь
!pip install -q -r $REPO/kaggle/requirements_gor.txt 2>&1 | tail -1
import torch, transformers, os; print(torch.__version__, transformers.__version__, torch.cuda.device_count())
assert transformers.__version__ == '5.21.0', transformers.__version__
M = {'B': ('facebook/dinov2-with-registers-base', 'a1d738ccfa7ae170945f210395d99dde8adb1805'),
     'L': ('facebook/dinov2-with-registers-large', 'e4c89a4e05589de9b3e188688a303d0f3c04d0f3')}
os.environ.update(DATA='/kaggle/input/train-expds-3', OUT='/kaggle/working/gp', DEVICE='cuda', THREADS='4')
os.makedirs('/kaggle/working/gp', exist_ok=True)

## A. Верификатор городищ: признаки виньеток (тот же протокол, что у DINOv2-S v3: 384 м + контекст 768 м, 224 px). Сначала smoke `LIMIT_FILES=2`.

In [ ]:
for t, (mid, rev) in M.items():
    os.environ.update(MODEL=mid, REVISION=rev, TAG=f'_reg{t}')
    !LIMIT_FILES=2 python $REPO/scripts/gor_patch/extract_feats.py 2>&1 | tail -2
import numpy as np; z = np.load('/kaggle/working/gp/feats_regL_test.npz')['F']; print(z.shape); assert z.shape[1] == 4 * 1024 and np.isfinite(z.astype('f4')).all()

In [ ]:
for t, (mid, rev) in M.items():
    os.environ.update(MODEL=mid, REVISION=rev, TAG=f'_reg{t}')
    !python $REPO/scripts/gor_patch/extract_feats.py 2>&1 | tail -2
!cd /kaggle/working/gp && rm -f *_test.* && tar czf /kaggle/working/gor_patch_reg.tgz feats_reg*.npz meta_reg*.json && ls -la /kaggle/working/gor_patch_reg.tgz

## B. Кроп-верификатор архитектуры Or (`scripts/crop_verify.py --encoder hf:...` — замороженный ViT + линейная проба, OOF по семьям, 10 фолдов)
Нужны STACK (стек до порогов, `scoreboard.py --dump-stack`) и GT (эталон трейна). Если их нет в Inputs — ячейка пропускается, сдать только часть A.
Контроль — тот же вызов с `--encoder resnet18` (дообучаемый, как раньше).

In [ ]:
STACK = '/kaggle/input/expds3-stack/stack_base.geojson'; GT = '/kaggle/input/expds3-stack/gt_train_v2.geojson'   # поправить
if os.path.exists(STACK) and os.path.exists(GT):
    for enc in ['resnet18', 'hf:%s@%s' % M['B'], 'hf:%s@%s' % M['L']]:
        tag = enc.split('/')[-1].split('@')[0] if enc.startswith('hf:') else enc
        !python $REPO/scripts/crop_verify.py --root $DATA --stack {STACK} --gt {GT} --markup or --classes arkhitektury \
            --res 0.5 --tile 96 --min-conf 0.05 --folds 10 --blend none --encoder "{enc}" \
            --out /kaggle/working/gp/verify_or_{tag}.geojson --report /kaggle/working/gp/verify_or_{tag}.json 2>&1 | tail -15
    !cd /kaggle/working/gp && tar czf /kaggle/working/verify_or_reg.tgz verify_or_*.json verify_or_*.geojson && ls -la /kaggle/working/verify_or_reg.tgz
else:
    print('STACK/GT нет — часть B пропущена')

Вернуть bot-5: `gor_patch_reg.tgz` (+ `verify_or_reg.tgz`, если была часть B) и хвосты логов.
На боксе: `tar xzf gor_patch_reg.tgz -C analysis/gor_patch` → `FEATS_TAG=_regB python scripts/gor_patch/gor_patch_eval.py` (и `_regL`) — сравнение с DINOv2-S по тем же фолдам и П1–П5.
Часть B оценивается табло: `scoreboard.py ... --rerank verify_or_<tag>.geojson --honest 10` (с `--blend geo` повторно на боксе/ПК из `verify_p`).